# 멀티에이전트 크롤링 파이프라인 구축

앞선 실습에서 우리는 두 가지 전문 에이전트를 설계했습니다.
- **Navigator** (03): 웹사이트 구조를 분석하고 CSS 셀렉터를 찾아내는 **탐색 전문가**
- **Coder** (04): Blueprint를 받아 실제로 동작하는 크롤링 코드를 작성·실행·디버깅하는 **개발 전문가**

이번 실습에서는 이 두 에이전트를 **멀티에이전트 워크플로우** 위에서 함께 작업하도록 통합합니다.

```
👤 사용자: "이 사이트에서 데이터 수집해줘"
    ↓
🗺️ Navigator ──(Blueprint JSON)──▶ 💻 Coder ──▶ 📄 수집 결과
```

Navigator와 Coder는 각자의 역할에만 집중하고, 두 에이전트 사이에는 **Blueprint(청사진)**라는 구조화된 JSON 객체가 오갑니다.
Blueprint에는 수집할 URL, 계층별 CSS 셀렉터, 페이지네이션 방식, JS 렌더링 여부 등이 담겨 있어,
Coder는 HTML을 직접 보지 않고도 정확한 크롤링 코드를 작성할 수 있습니다.

### 이번 실습에서 배우는 것

| Part | 내용 | 핵심 학습 |
|:---|:---|:---|
| **Part 1** | Blueprint 스키마 복습 + 검증 예시 | Pydantic 구조화 출력, 스키마 설계 |
| **Part 2** | Navigator + Coder 수동 연결 | 에이전트 간 데이터 전달 패턴 |
| **Part 3** | 멀티에이전트 패턴 비교 | Handoff vs Agent as Tool |
| **Part 4** | 전체 파이프라인 자동화 | 함수 기반 오케스트레이션 |

.env 파일을 로드하여 환경변수에 등록합니다. 노트북을 위한 실행 경로를 세팅합니다.

In [ ]:
import os, sys
from dotenv import load_dotenv

load_dotenv(override=True)

# PROJECT_ROOT를 .env에서 읽기 (없으면 현재 디렉토리)
project_root = os.getenv("PROJECT_ROOT", os.getcwd())

# 프로젝트 루트가 유효하지 않으면, 현재 위치에서 상위로 찾기
if not os.path.exists(os.path.join(project_root, "app")):
    current = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(current, "app")):
            project_root = current
            break
        current = os.path.dirname(current)

# Working Directory 설정
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import app

print(f"✅ Project Root: {project_root}")
print(f"✅ Working Directory: {os.getcwd()}")

In [ ]:
import asyncio
from browser_use import Agent, Browser, ChatGoogle

import nest_asyncio
nest_asyncio.apply()

In [ ]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Multiagent_workflow" 

---
## Part 1: Blueprint 스키마 복습

### 왜 Blueprint가 필요한가?

사람이 웹 스크래핑을 할 때를 생각해봅시다:
1. 먼저 사이트를 **둘러보고** 구조를 파악합니다 (어디에 데이터가 있는지?)
2. 그 다음 **코드를 짜서** 데이터를 수집합니다

이 두 단계는 요구하는 능력이 완전히 다릅니다:
- 1단계: 브라우저를 조작하고 HTML을 읽는 능력 → **Navigator**
- 2단계: Python 코드를 작성하고 디버깅하는 능력 → **Coder**

Navigator가 "이 사이트는 이런 구조야, 이 셀렉터를 쓰면 돼"라고 
구조화된 JSON으로 정리하면, Coder는 사이트를 직접 보지 않고도 
정확한 크롤링 코드를 작성할 수 있습니다.

### 클래스 구조

03, 04 노트북에서 이미 다룬 `app/schemas.py`의 Blueprint 모델을 다시 확인합니다.

```
NavigatorBlueprintCollection  ← Navigator의 최종 출력
└── blueprints: list[NavigatorBlueprint]
    └── layers: list[PageLayer]  ← 탐색 계층 단위 블록
```

| 클래스 | 핵심 필드 |
|:---|:---|
| `PageLayer` | `layer_name`, `url_pattern`, `container_selector`, `selectors`, `navigate_to_next`, `pagination_method` |
| `NavigatorBlueprint` | `entry_urls`, `total_layers`, `layers`, `rendering_type`, `anti_bot_notes` |
| `NavigatorBlueprintCollection` | `total_jobs`, `blueprints` |

In [ ]:
from app.schemas import (
    NavigatorContext, NavigatorBlueprintCollection,
    NavigatorBlueprint, PageLayer,
    SeniorCoderContext
)

print("✅ 스키마 임포트 완료")

### 스키마 검증 예시 3가지

| 예시 | 설명 |
|:---|:---|
| 예시 1 | Blueprint 1개, `entry_urls` 1개 (단일 섹션) |
| 예시 2 | Blueprint 1개, `entry_urls` 2개 (구조 동일, URL만 다름) |
| 예시 3 | Blueprint 2개 (사이트 구조가 달라 분리) |

In [ ]:
import json

# 예시 1: 단일 섹션 (네이버 금융 주식 시세) - Blueprint 1개, entry_urls 1개
sample_single = NavigatorBlueprintCollection(
    total_jobs=1,
    blueprints=[
        NavigatorBlueprint(
            entry_urls=["https://finance.naver.com/sise/sise_rise.nhn"],
            total_layers=1,
            layers=[
                PageLayer(
                    layer_name="주식 상승 종목 테이블",
                    url_pattern="https://finance.naver.com/sise/sise_rise.nhn",
                    selectors={"stock_name": "td.col_type1 a", "change_rate": "td.rate_t"},
                    navigate_to_next=None,
                    pagination_method="URL파라미터"
                )
            ],
            rendering_type="Static SSR",
            anti_bot_notes="없음"
        )
    ]
)

# 예시 2: 동일 구조, 출발 URL만 다른 경우 (정치면 + 사회면) - Blueprint 1개, entry_urls 2개
sample_multi_url = NavigatorBlueprintCollection(
    total_jobs=1,
    blueprints=[
        NavigatorBlueprint(
            entry_urls=[
                "https://news.naver.com/section/100",  # 정치
                "https://news.naver.com/section/102",  # 사회
            ],
            total_layers=2,
            layers=[
                PageLayer(
                    layer_name="뉴스 목록",
                    url_pattern="https://news.naver.com/section/{sid}",
                    selectors={"article_link": "a.sa_text_title"},
                    navigate_to_next="a.sa_text_title",
                    pagination_method="AJAX버튼"
                ),
                PageLayer(
                    layer_name="기사 상세",
                    url_pattern="https://n.news.naver.com/mnews/article/{mid}/{aid}",
                    selectors={"title": "h2.media_end_head_headline", "body": "#dic_area", "datetime": ".media_end_head_info_dateline"},
                    navigate_to_next=None
                )
            ],
            rendering_type="Dynamic CSR/JS",
            anti_bot_notes="더보기 버튼 AJAX 방식"
        )
    ]
)

# 예시 3: 구조가 다른 두 사이트 - Blueprint 2개
sample_multi_blueprint = NavigatorBlueprintCollection(
    total_jobs=2,
    blueprints=[
        NavigatorBlueprint(
            entry_urls=["https://news.naver.com/section/100"],
            total_layers=2,
            layers=[
                PageLayer(layer_name="뉴스 목록", url_pattern="https://news.naver.com/section/100",
                          selectors={"article_link": "a.sa_text_title"}, navigate_to_next="a.sa_text_title"),
                PageLayer(layer_name="기사 상세", url_pattern="https://n.news.naver.com/mnews/article/{mid}/{aid}",
                          selectors={"title": "h2.media_end_head_headline", "body": "#dic_area"}, navigate_to_next=None)
            ],
            rendering_type="Dynamic CSR/JS", anti_bot_notes="없음"
        ),
        NavigatorBlueprint(
            entry_urls=["https://www.danawa.com/product/?cateCode=112758"],
            total_layers=2,
            layers=[
                PageLayer(layer_name="상품 목록", url_pattern="https://www.danawa.com/product/?cateCode=112758",
                          selectors={"product_link": "a.prod_name"}, navigate_to_next="a.prod_name", pagination_method="URL파라미터"),
                PageLayer(layer_name="상품 상세", url_pattern="https://prod.danawa.com/info/?pcode={id}",
                          selectors={"name": "h3.prod_tit", "price": "span.low_price", "specs": "div.spec_list"}, navigate_to_next=None)
            ],
            rendering_type="Dynamic CSR/JS", anti_bot_notes="없음"
        )
    ]
)

print("✅ 예시 1 (단일):")
print(sample_single.model_dump_json(indent=2))

In [ ]:
print("\n✅ 예시 2 (동일 구조, URL 2개):")
print(sample_multi_url.model_dump_json(indent=2))

In [ ]:
print("\n✅ 예시 3 (다른 구조 사이트 2개):")
print(sample_multi_blueprint.model_dump_json(indent=2))

---
## Part 2: Navigator → Coder 수동 연결

### Step 1: Navigator 에이전트로 Blueprint 생성

`app/agents/navigator.py`에서 Navigator 에이전트를 임포트합니다.
Navigator는 `response_mode="blueprint"` 컨텍스트로 실행하면 **구조화된 Blueprint**를 출력합니다.

In [ ]:
from app.agents import create_navigator_agent
from app.tools.common import ARTIFACT_DIR
from browser_use import Browser
from langchain_core.messages import HumanMessage

navigator_agent = create_navigator_agent()
print(f"✅ Navigator 에이전트 생성 완료")
print(f"📂 Artifact 디렉토리: {ARTIFACT_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────
# Navigator 실행: 네이버 뉴스 정치 섹션 기사 수집 Blueprint 생성
# ──────────────────────────────────────────────────────

# Runtime Context용 공유 브라우저 인스턴스 생성
shared_browser = Browser(
    headless=False,
    disable_security=True,
    keep_alive=True,
)

# Thread ID 설정 및 브라우저를 Context에 주입
nav_config = {"configurable": {"thread_id": "nav_pipeline_test"}}
nav_context = NavigatorContext(shared_browser=shared_browser, response_mode="blueprint")

# Navigator Agent 실행 (Blueprint 생성 지시)
nav_response = await navigator_agent.ainvoke(
    {"messages": [HumanMessage(
        "https://news.naver.com에서 정치 섹션의 "
        "최신 기사 제목 5개와 각 기사의 상세 URL을 수집하는 Blueprint를 치밀하게 작성해 주세요. "
        "탐색은 이 URL에서 시작하되, 텍스트(제목)와 링크(URL)의 셀렉터 속성이 명확히 분리된 "
        "완벽한 Blueprint를 설계하는 것이 목표입니다."
    )]},
    config=nav_config,
    context=nav_context,
)

# 구조화된 Blueprint 추출
blueprint_collection: NavigatorBlueprintCollection = nav_response["structured_response"]

print(f"\n✅ 결과: {blueprint_collection.total_jobs}개 Blueprint 생성")
for i, bp in enumerate(blueprint_collection.blueprints):
    print(f"\n  [Blueprint {i+1}]")
    print(f"    entry_urls : {bp.entry_urls}")
    print(f"    계층 수    : {bp.total_layers}")
    print(f"    렌더링 방식: {bp.rendering_type}")
    print(f"    안티봇 주의: {bp.anti_bot_notes}")
    for j, layer in enumerate(bp.layers):
        print(f"    Layer {j+1} ({layer.layer_name}) | selectors={layer.selectors}")

# 공유 브라우저 정리
await shared_browser.stop()
print("\n🛑 공유 브라우저 세션 종료 완료")

#### Blueprint 저장

Navigator가 생성한 Blueprint를 JSON 파일로 저장합니다.
이 파일이 Navigator → Coder 사이의 **데이터 계약(Contract)** 역할을 합니다.

In [ ]:
# Blueprint를 JSON 파일로 저장
def save_blueprints(collection: NavigatorBlueprintCollection, prefix: str = "blueprint"):
    """Blueprint 컬렉션을 개별 JSON 파일로 저장"""
    paths = []
    for i, bp in enumerate(collection.blueprints):
        filename = f"{prefix}_{i+1}.json"
        filepath = os.path.join(ARTIFACT_DIR, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(bp.model_dump(), f, ensure_ascii=False, indent=2)
        print(f"  📁 저장: {filepath}")
        paths.append(filepath)
    return paths

print("📁 Blueprint 파일 저장:")
blueprint_paths = save_blueprints(blueprint_collection, prefix="pipeline_test")

### Step 2: Coder 에이전트로 크롤링 코드 생성 + 실행

Navigator가 저장한 Blueprint JSON을 읽어서 Coder에게 전달합니다.
Coder는 Blueprint를 해석하고, 크롤링 코드를 작성·실행·디버깅합니다.

In [ ]:
from app.agents import create_coder_agent

coder_agent = create_coder_agent()
print("✅ Coder 에이전트 생성 완료")

In [ ]:
# ──────────────────────────────────────────────────────
# Coder 실행: Blueprint를 받아 크롤링 코드 작성 + 실행
# ──────────────────────────────────────────────────────

# Navigator가 저장한 Blueprint 읽기
blueprint_path = blueprint_paths[0]  # 첫 번째 Blueprint 사용
with open(blueprint_path, "r", encoding="utf-8") as f:
    blueprint_content = f.read()

print(f"📄 Blueprint 파일: {blueprint_path}")
print(f"📄 Navigator의 Blueprint를 획득했습니다. Coder에게 전달합니다...\n")

# Coder에게 명령 전달
mission_query = f"""
다음은 Navigator 에이전트가 탐색하여 넘겨준 크롤링 설계도(Blueprint) JSON 입니다.

[크롤링 Blueprint JSON]
{blueprint_content}

이걸 바탕으로 데이터를 수집해주세요.
"""

coder_config = {"configurable": {"thread_id": "coder_pipeline_test"}}
coder_context = SeniorCoderContext()

print("⏳ Coder 가동 중 (코드 작성 및 실행)... 수십 초가 소요될 수 있습니다.")
coder_result = await coder_agent.ainvoke(
    {"messages": [HumanMessage(content=mission_query)]},
    config=coder_config,
    context=coder_context
)

# 최종 응답 출력
final_response = coder_result['messages'][-1].content
print("\n" + "=" * 60)
print("✅ Coder 작업 완료 보고서:")
print("=" * 60)
print(final_response[0]['text'])

In [ ]:
for m in coder_result['messages']:
    m.pretty_print()

### 🎯 수동 연결 정리

위에서 우리는 다음과 같이 **수동으로** Navigator와 Coder를 연결했습니다:

```python
# 수동 연결 방식
nav_response = await navigator_agent.ainvoke(...)     # 1단계: Navigator 실행
blueprint = nav_response["structured_response"]       # 중간 결과 수동 추출
save_blueprints(blueprint, ...)                       # Blueprint 파일 저장
coder_result = await coder_agent.ainvoke(...)          # 2단계: Coder 실행
```

이 방식은 간단하지만 한계가 있습니다:
- 사람이 중간에 개입해야 다음 단계가 실행됩니다
- 새로운 에이전트가 추가되면 연결 코드가 계속 늘어납니다
- 조건에 따른 분기(예: 실패 시 재시도)를 구현하기가 복잡합니다

**Part 3**에서는 멀티에이전트 패턴을 학습하고, **Part 4**에서 이를 자동화합니다.

---
## Part 3: 전체 파이프라인 자동화

수동 연결을 하나의 함수로 통합합니다.
사용자가 **URL과 수집 목표만 입력**하면, Navigator → Coder가 자동으로 실행됩니다.

```
[사용자 입력: URL + 수집 목표]
        ↓
[Navigator] → Blueprint 생성 → JSON 저장
        ↓
[Coder] → Blueprint 해석 → 코드 작성 → 실행 → 데이터 수집
        ↓
[결과: 수집된 데이터 파일]
```

In [ ]:
from app.agents import create_navigator_agent, create_coder_agent
from app.schemas import NavigatorContext, NavigatorBlueprintCollection, SeniorCoderContext
from app.tools.common import ARTIFACT_DIR
from browser_use import Browser
from langchain_core.messages import HumanMessage
import json, os

async def run_crawling_pipeline(user_query: str, headless: bool = False):
    """
    사용자의 수집 목표를 받아 Navigator → Coder 파이프라인을 자동 실행합니다.
    
    Args:
        user_query: 수집 목표 (예: "https://news.naver.com 정치 기사 5개")
        headless: 브라우저를 숨길지 여부 (True: 숨김, False: 보임)
    
    Returns:
        dict: {"blueprint": ..., "coder_response": ..., "files": ...}
    """
    print("=" * 60)
    print("🚀 멀티에이전트 크롤링 파이프라인 시작")
    print("=" * 60)
    print(f"📋 목표: {user_query}\n")
    
    # ──────────────────────────────────────────────────
    # Phase 1: Navigator — 사이트 분석 + Blueprint 생성
    # ──────────────────────────────────────────────────
    print("─" * 40)
    print("🗺️  Phase 1: Navigator 에이전트 가동")
    print("─" * 40)
    
    navigator = create_navigator_agent()
    shared_browser = Browser(
        headless=headless,
        disable_security=True,
        keep_alive=True,
    )
    
    nav_config = {"configurable": {"thread_id": "pipeline_nav"}}
    nav_context = NavigatorContext(
        shared_browser=shared_browser,
        response_mode="blueprint"
    )
    
    try:
        nav_response = await navigator.ainvoke(
            {"messages": [HumanMessage(user_query)]},
            config=nav_config,
            context=nav_context,
        )
        blueprint_collection: NavigatorBlueprintCollection = nav_response["structured_response"]
        
        print(f"\n✅ Navigator 완료: {blueprint_collection.total_jobs}개 Blueprint 생성")
        
        # Blueprint 저장
        blueprint_paths = []
        for i, bp in enumerate(blueprint_collection.blueprints):
            filename = f"auto_pipeline_{i+1}.json"
            filepath = os.path.join(ARTIFACT_DIR, filename)
            with open(filepath, "w", encoding="utf-8") as f:
                json.dump(bp.model_dump(), f, ensure_ascii=False, indent=2)
            print(f"  📁 저장: {filepath}")
            blueprint_paths.append(filepath)
    finally:
        await shared_browser.stop()
        print("🛑 Navigator 브라우저 종료")
    
    # ──────────────────────────────────────────────────
    # Phase 2: Coder — Blueprint 기반 코드 생성 + 실행
    # ──────────────────────────────────────────────────
    print(f"\n{'─' * 40}")
    print("💻 Phase 2: Coder 에이전트 가동")
    print("─" * 40)
    
    coder = create_coder_agent()
    all_results = []
    
    for idx, bp_path in enumerate(blueprint_paths):
        print(f"\n  📄 Blueprint {idx+1}/{len(blueprint_paths)}: {bp_path}")
        
        with open(bp_path, "r", encoding="utf-8") as f:
            bp_content = f.read()
        
        mission = f"""
다음은 Navigator 에이전트가 탐색하여 넘겨준 크롤링 설계도(Blueprint) JSON 입니다.

[크롤링 Blueprint JSON]
{bp_content}

이걸 바탕으로 데이터를 수집해주세요.
"""
        
        coder_config = {"configurable": {"thread_id": f"pipeline_coder_{idx}"}}
        coder_context = SeniorCoderContext()
        
        result = await coder.ainvoke(
            {"messages": [HumanMessage(content=mission)]},
            config=coder_config,
            context=coder_context
        )
        
        final_msg = result['messages'][-1].content
        all_results.append(final_msg)
        print(f"  ✅ Blueprint {idx+1} 처리 완료")
    
    # ──────────────────────────────────────────────────
    # 최종 결과 요약
    # ──────────────────────────────────────────────────
    print(f"\n{'=' * 60}")
    print("🏁 파이프라인 완료!")
    print("=" * 60)
    print(f"  Blueprint 수: {len(blueprint_paths)}")
    print(f"  결과 파일 위치: {ARTIFACT_DIR}")
    
    return {
        "blueprint_collection": blueprint_collection,
        "blueprint_paths": blueprint_paths,
        "coder_results": all_results
    }

print("✅ run_crawling_pipeline 함수 정의 완료")

### 🧪 파이프라인 실행 테스트

아래 셀에서 수집 목표를 입력하고 전체 파이프라인을 실행합니다.

In [ ]:
# 🚀 전체 파이프라인 실행
result = await run_crawling_pipeline(
    user_query=(
        "https://news.naver.com에서 정치 섹션의 "
        "최신 기사 제목 5개와 각 기사의 상세 URL을 수집하는 Blueprint를 치밀하게 작성해 주세요."
    ),
    headless=False
)

# 결과 확인
print("\n📊 Coder 최종 보고서:")
for i, r in enumerate(result["coder_results"]):
    print(f"\n--- Blueprint {i+1} 결과 ---")
    print(r[:500])  # 첫 500자만 표시

### 🏆 완성된 파이프라인 전체 흐름

```
HumanMessage (URL + 목표)
    ↓
🗺️ Navigator 에이전트
    ├─ get_page_structure() → CSS 셀렉터 후보 추출
    ├─ verify_selectors() → 실제 브라우저에서 셀렉터 검증
    └─ browse_web() (필요 시) → 동적 페이지 Fallback
    ↓
NavigatorBlueprintCollection (JSON)
    { selectors, rendering_type, pagination_method, ... }
    ↓
💻 Coder 에이전트
    ├─ Blueprint 해석
    ├─ 라이브러리 선택 (requests/bs4 or playwright)
    ├─ create_new_file() → 크롤링 코드 작성
    ├─ run_python_script() → 테스트 실행
    ├─ validate_collected_data() → 품질 검증
    └─ edit_code_file() → 수량 확대 후 재실행 (필요 시)
    ↓
📄 scraping_result.json ✅
```

### 📦 모듈 구조 정리

```
app/
├── agents/
│   ├── navigator.py     ← create_navigator_agent()
│   ├── coder.py         ← create_coder_agent()
│   └── utils.py         ← dynamic_response_format 미들웨어
├── tools/
│   ├── navigator.py     ← get_page_structure, verify_selectors, browse_web
│   ├── coder.py         ← read/edit/create/run/validate 도구 6개
│   └── common.py        ← ARTIFACT_DIR, CostTracker
├── prompts/
│   ├── navigator.py     ← NAVIGATOR_SYSTEM_PROMPT, BROWSER_AGENT_GUIDE
│   └── coder.py         ← CODER_SYSTEM_PROMPT
└── schemas.py           ← Blueprint 모델, NavigatorContext, SeniorCoderContext
```


---
## Part 4: 멀티에이전트 패턴 비교

> 📖 자세한 내용은 `reference/LangChain/multi_agent_patterns.md`를 참고하세요.

LangChain에서 감독형 다중 에이전트 팀을 구축하는 두 가지 대표적인 패턴을 비교합니다.

### Handoff vs Agent as Tool

| 비교 항목 | **Handoff 방식** | **Agent as Tool 방식** |
|:---|:---|:---|
| **핵심 철학** | 맥락의 공유 (Shared Context) | 맥락의 격리 (Context Isolation) |
| **데이터 전달** | 전체 대화 기록을 그대로 전달 | 필요한 정보만 선별하여 전달 |
| **제어 흐름** | 단방향 이동 (A→B로 제어권 위임) | 왕복 호출 (관리자가 호출하고 결과를 받음) |
| **토큰 효율성** | 낮음 (대화가 길어질수록 비용 증가) | 높음 (하위 에이전트는 최소 입력만 받음) |
| **비유** | 릴레이 달리기: 바통(전체 문맥)을 넘김 | 팀장과 팀원: "엑셀만 정리해와" 쪽지를 줌 |


### 🔜 다음 단계

지금까지 우리는 실습에서 **Navigator → Coder 파이프라인**을 순서대로 수동 연결했습니다.
한 단계 더 나아가, 이 두 에이전트를 **한 명의 감독자(Supervisor)가 지휘하는 팀**으로 재구성할 수 있습니다.

이 방식은 멀티에이전트 워크플로우보다 더 유연한 작업처리가 가능합니다.